# NB-R04 — Dependence-Aware Statistical Testing

**Pipeline stage:** 4 of 13

**Purpose.** Test whether the High-VIX vs Low-VIX accuracy difference is statistically distinguishable from chance, using inference procedures that account for the serial correlation induced by the 21-day-ahead forecast horizon.

**Why this matters.** Because the target is a 21-day-ahead label, consecutive daily observations share up to 20 overlapping future days, which violates the independence assumption behind a standard Fisher's exact test and inflates apparent significance. This notebook instead reports: (a) Fisher's exact test on a non-overlapping subsample (every 21st observation), (b) a circular block-bootstrap confidence interval for the accuracy difference (block length 21), and (c) a block-permutation test, all with Bonferroni correction across the three forecast horizons tested in this study.

**Inputs:** `data/processed/test_predictions.csv` (from NB-R03).

**Outputs:** `results/table4_regime_metrics.csv`, `results/table5_statistical_tests.csv`, `results/statistical_tests.json`, `plots/R04_statistical_tests.png`.

**Result:** a 33.5-percentage-point observed accuracy gap (100.0% High-VIX vs 66.5% Low-VIX), but neither the non-overlapping Fisher test nor the block-permutation test rejects the null hypothesis after Bonferroni correction (both p = 1.000).


In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import fisher_exact
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    precision_score, recall_score, f1_score, confusion_matrix
)

PROJ    = Path('..').resolve()  # repo root, assuming this notebook is run from notebooks/
PROC    = PROJ / 'data' / 'processed'
RESULTS = PROJ / 'results'
PLOTS   = PROJ / 'plots'

SEED    = 42
N_BOOT  = 10000
BLOCK_LEN = 21   # matches prediction horizon
ALPHA   = 0.05
N_HORIZONS = 3   # Bonferroni: 1d, 5d, 21d

rng = np.random.default_rng(SEED)

test = pd.read_csv(PROC / 'test_predictions.csv', parse_dates=['date'])
print(f'Test rows: {len(test)}')
print('Regime counts:')
print(test['regime_fixed'].value_counts())

## 1. Overall Accuracy and Majority-Class Baseline

In [ ]:
y_true = test['dir_21d'].values
y_pred = test['stack_pred'].values
y_prob = test['stack_prob'].values
regime = test['regime_fixed'].values

# Majority-class baseline (always predict Up since Up > 50%)
majority_class = int(np.round(y_true.mean()))
y_majority = np.full_like(y_true, majority_class)

overall_acc     = accuracy_score(y_true, y_pred)
majority_acc    = accuracy_score(y_true, y_majority)
overall_auc     = roc_auc_score(y_true, y_prob)

print(f'Test period Up%:              {y_true.mean()*100:.1f}%')
print(f'Majority-class baseline acc:  {majority_acc*100:.1f}%  (always predict {majority_class})')
print(f'Stack ensemble accuracy:      {overall_acc*100:.1f}%')
print(f'Stack ensemble ROC-AUC:       {overall_auc:.4f}')

## 2. Regime-Stratified Metrics
Reports accuracy, precision, recall, F1, and majority-class baseline **within each regime**.

In [ ]:
regime_results = {}

for reg in ['High-VIX', 'Low-VIX']:
    mask = regime == reg
    yt = y_true[mask]
    yp = y_pred[mask]
    ypr = y_prob[mask]

    up_pct   = yt.mean() * 100
    maj_class = int(np.round(yt.mean()))
    maj_acc   = max(yt.mean(), 1 - yt.mean()) * 100   # majority always predicts dominant class
    acc       = accuracy_score(yt, yp) * 100
    prec      = precision_score(yt, yp, zero_division=0) * 100
    rec       = recall_score(yt, yp, zero_division=0) * 100
    f1        = f1_score(yt, yp, zero_division=0) * 100
    auc       = roc_auc_score(yt, ypr) if len(np.unique(yt)) > 1 else np.nan
    cm        = confusion_matrix(yt, yp)

    regime_results[reg] = {
        'n': int(mask.sum()), 'up_pct': up_pct, 'majority_baseline_acc': maj_acc,
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc,
        'confusion_matrix': cm.tolist()
    }

    print(f'\n--- {reg} (n={mask.sum()}) ---')
    print(f'  Up%:                   {up_pct:.1f}%')
    print(f'  Majority baseline acc: {maj_acc:.1f}%  (always predict {maj_class})')
    print(f'  Stack accuracy:        {acc:.1f}%')
    print(f'  Precision:             {prec:.1f}%')
    print(f'  Recall:                {rec:.1f}%')
    print(f'  F1:                    {f1:.1f}%')
    print(f'  ROC-AUC:               {auc:.4f}')
    if cm.shape == (2, 2):
        print(f'  Confusion matrix:\\n    [[TN={cm[0,0]}, FP={cm[0,1]}], [FN={cm[1,0]}, TP={cm[1,1]}]]')
    else:
        print(f'  Confusion matrix (single class): {cm.tolist()}')

acc_diff = regime_results['High-VIX']['accuracy'] - regime_results['Low-VIX']['accuracy']
print(f'\nAccuracy difference (High - Low): {acc_diff:.1f} pp')

## 3. Fisher's Exact Test on Non-Overlapping Observations

In [ ]:
def fisher_non_overlapping(y_true, y_pred, regime, step=21):
    """Fisher's exact test using every `step`-th observation to avoid serial correlation."""
    idx = np.arange(0, len(y_true), step)
    yt  = y_true[idx]
    yp  = y_pred[idx]
    reg = regime[idx]

    results = {}
    for reg_label in ['High-VIX', 'Low-VIX']:
        m = reg == reg_label
        correct   = int((yp[m] == yt[m]).sum())
        incorrect = int((yp[m] != yt[m]).sum())
        results[reg_label] = (correct, incorrect)

    high_c, high_i = results['High-VIX']
    low_c,  low_i  = results['Low-VIX']
    contingency = [[high_c, high_i], [low_c, low_i]]
    odds, p = fisher_exact(contingency, alternative='greater')

    n_eff = len(idx)
    print(f'Non-overlapping sample (every {step}th day): n_eff={n_eff}')
    print(f'  High-VIX: {high_c} correct, {high_i} incorrect')
    print(f'  Low-VIX:  {low_c} correct, {low_i} incorrect')
    print(f'  Fisher p (one-sided): {p:.4e}')
    print(f'  Bonferroni-corrected p (x{N_HORIZONS}): {min(p * N_HORIZONS, 1.0):.4e}')
    return p, contingency

fisher_p, contingency = fisher_non_overlapping(y_true, y_pred, regime, step=BLOCK_LEN)

## 4. Block Bootstrap Confidence Interval for Accuracy Difference

In [ ]:
def block_bootstrap_acc_diff(y_true, y_pred, regime, n_boot=10000, block_len=21, rng=None):
    """
    Circular block bootstrap for the accuracy difference (High-VIX acc - Low-VIX acc).
    Block length = 21 to account for 21-day prediction horizon overlaps.
    """
    if rng is None:
        rng = np.random.default_rng(42)

    n = len(y_true)
    n_blocks = int(np.ceil(n / block_len))

    boot_diffs = []
    for _ in range(n_boot):
        # Sample block start indices with replacement
        starts = rng.integers(0, n, size=n_blocks)
        boot_idx = np.concatenate([np.arange(s, s + block_len) % n for s in starts])[:n]

        yt_b  = y_true[boot_idx]
        yp_b  = y_pred[boot_idx]
        reg_b = regime[boot_idx]

        high_mask = reg_b == 'High-VIX'
        low_mask  = reg_b == 'Low-VIX'

        if high_mask.sum() < 2 or low_mask.sum() < 2:
            continue

        acc_h = accuracy_score(yt_b[high_mask], yp_b[high_mask])
        acc_l = accuracy_score(yt_b[low_mask],  yp_b[low_mask])
        boot_diffs.append(acc_h - acc_l)

    boot_diffs = np.array(boot_diffs)
    ci_lo = np.percentile(boot_diffs, 2.5)
    ci_hi = np.percentile(boot_diffs, 97.5)
    observed_diff = (regime_results['High-VIX']['accuracy'] - regime_results['Low-VIX']['accuracy']) / 100

    print(f'Block bootstrap (n_boot={len(boot_diffs)}, block_len={block_len}):')
    print(f'  Observed difference:     {observed_diff*100:.1f} pp')
    print(f'  95% CI (block boot):     [{ci_lo*100:.1f}, {ci_hi*100:.1f}] pp')
    print(f'  CI includes zero?        {ci_lo <= 0 <= ci_hi}')
    return boot_diffs, ci_lo, ci_hi

boot_diffs, ci_lo, ci_hi = block_bootstrap_acc_diff(y_true, y_pred, regime,
                                                      n_boot=N_BOOT, block_len=BLOCK_LEN, rng=rng)

## 5. Block Permutation Test

In [ ]:
def block_permutation_test(y_true, y_pred, regime, n_perm=10000, block_len=21, rng=None):
    """
    Block permutation test for the null hypothesis that accuracy is equal across regimes.
    Shuffles regime labels in blocks of `block_len` to preserve autocorrelation structure.
    """
    if rng is None:
        rng = np.random.default_rng(42)

    high_acc_obs = regime_results['High-VIX']['accuracy'] / 100
    low_acc_obs  = regime_results['Low-VIX']['accuracy']  / 100
    observed_diff = high_acc_obs - low_acc_obs

    n = len(regime)
    perm_diffs = []

    for _ in range(n_perm):
        # Block shuffle: split regime into blocks and shuffle block order
        blocks = [regime[i:i+block_len] for i in range(0, n, block_len)]
        perm_blocks = rng.permutation(len(blocks))
        perm_regime = np.concatenate([blocks[i] for i in perm_blocks])[:n]

        high_m = perm_regime == 'High-VIX'
        low_m  = perm_regime == 'Low-VIX'
        if high_m.sum() < 2 or low_m.sum() < 2:
            continue

        acc_h = accuracy_score(y_true[high_m], y_pred[high_m])
        acc_l = accuracy_score(y_true[low_m],  y_pred[low_m])
        perm_diffs.append(acc_h - acc_l)

    perm_diffs = np.array(perm_diffs)
    p_perm = (perm_diffs >= observed_diff).mean()

    print(f'Block permutation test (n_perm={len(perm_diffs)}, block_len={block_len}):')
    print(f'  Observed difference:         {observed_diff*100:.1f} pp')
    print(f'  Permutation p-value:         {p_perm:.4e}')
    print(f'  Bonferroni-corrected (x{N_HORIZONS}): {min(p_perm * N_HORIZONS, 1.0):.4e}')
    return perm_diffs, p_perm

perm_diffs, p_perm = block_permutation_test(y_true, y_pred, regime,
                                              n_perm=N_BOOT, block_len=BLOCK_LEN, rng=rng)

## 6. Summary Table and Plots

In [ ]:
stats_summary = {
    'high_vix': regime_results['High-VIX'],
    'low_vix':  regime_results['Low-VIX'],
    'accuracy_diff_pp': acc_diff,
    'fisher_non_overlap_p': float(fisher_p),
    'fisher_bonferroni_p': float(min(fisher_p * N_HORIZONS, 1.0)),
    'block_boot_ci_95': [float(ci_lo * 100), float(ci_hi * 100)],
    'block_perm_p': float(p_perm),
    'block_perm_bonferroni_p': float(min(p_perm * N_HORIZONS, 1.0)),
    'majority_baseline_overall': float(max(y_true.mean(), 1 - y_true.mean()) * 100),
    'stack_overall_acc': float(overall_acc * 100),
    'stack_overall_auc': float(overall_auc)
}

with open(RESULTS / 'statistical_tests.json', 'w') as f:
    json.dump(stats_summary, f, indent=2, default=str)

print('Statistical test summary:')
print(f'  High-VIX accuracy:          {regime_results["High-VIX"]["accuracy"]:.1f}%')
print(f'  Low-VIX accuracy:           {regime_results["Low-VIX"]["accuracy"]:.1f}%')
print(f'  Difference:                 {acc_diff:.1f} pp')
print(f'  Block bootstrap 95% CI:     [{ci_lo*100:.1f}, {ci_hi*100:.1f}] pp')
print(f'  Fisher p (non-overlapping): {fisher_p:.3e}')
print(f'  Block permutation p:        {p_perm:.3e}')
print('Results saved to statistical_tests.json')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Block bootstrap distribution
axes[0].hist(boot_diffs * 100, bins=60, alpha=0.7, color='steelblue')
observed_diff_pp = (regime_results['High-VIX']['accuracy'] - regime_results['Low-VIX']['accuracy'])
axes[0].axvline(observed_diff_pp, color='red', linewidth=2, label=f'Observed={observed_diff_pp:.1f}pp')
axes[0].axvline(ci_lo * 100, color='orange', linewidth=1.5, linestyle='--', label=f'95% CI lo={ci_lo*100:.1f}pp')
axes[0].axvline(ci_hi * 100, color='orange', linewidth=1.5, linestyle='--', label=f'95% CI hi={ci_hi*100:.1f}pp')
axes[0].axvline(0, color='black', linewidth=1, linestyle=':')
axes[0].set_title('Block Bootstrap: Accuracy Difference (High - Low VIX)')
axes[0].set_xlabel('Accuracy Difference (pp)')
axes[0].legend(fontsize=8)

# Right: Block permutation null distribution
axes[1].hist(perm_diffs * 100, bins=60, alpha=0.7, color='gray')
axes[1].axvline(observed_diff_pp, color='red', linewidth=2, label=f'Observed={observed_diff_pp:.1f}pp')
axes[1].set_title('Block Permutation Null Distribution')
axes[1].set_xlabel('Accuracy Difference (pp)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(PLOTS / 'R04_statistical_tests.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

---
## Summary

**Pipeline stage:** 4 of 13 (see `notebooks/README.md` for the full pipeline map).

**Artifacts produced by this notebook:**

- `results/table4_regime_metrics.csv`
- `results/table5_statistical_tests.csv`
- `results/statistical_tests.json`
- `plots/R04_statistical_tests.png`

**Next notebook:** `NB-R05_walkforward_generalizability.ipynb`
